<a href="https://colab.research.google.com/github/SebGalvaleira/mineria-datos-ii-parcial/blob/main/notebooks/e2/E2_04_gold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E2 · 04 · Gold: mart FinOps `org_daily_usage_by_service`
Agrega Silver al grano organización × día × servicio. Se recalculan fechas completas desde Silver y se reemplazan
sus particiones: correcto ante eventos tardíos e idempotente ante reprocesos. Se valida contra el MapReduce de la entrega 1.

In [1]:
!pip install -q "pyspark>=4.0,<4.1"
import os, shutil, subprocess
os.environ.pop('JAVA_HOME', None)

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
LAKE_DRIVE = Path('/content/drive/MyDrive/cpa_datalake')   # respaldo persistente
LAKE       = Path('/content/cpa_datalake')                  # copia de trabajo (disco local de Colab)

RUTAS = {z: LAKE / z for z in ['landing', 'stream_input', 'bronze', 'silver',
                               'gold', 'quarantine', 'checkpoints', 'logs']}

def restaurar(*rutas):
    """Copia desde Drive SOLO las carpetas que este notebook necesita (si no están ya en local)."""
    for r in rutas:
        origen, destino = LAKE_DRIVE / r, LAKE / r
        if origen.exists() and not destino.exists():
            shutil.copytree(origen, destino)
            print('✓ restaurado:', r)

def guardar_en_drive(*rutas):
    """Sincroniza en Drive SOLO las carpetas indicadas."""
    for r in rutas:
        origen, destino = LAKE / r, LAKE_DRIVE / r
        destino.mkdir(parents=True, exist_ok=True)
        subprocess.run(['rsync', '-a', '--delete', f'{origen}/', f'{destino}/'], check=True)
        print('✓ sincronizado:', r)

from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder.appName('cloud-provider-analytics')
         .config('spark.sql.session.timeZone', 'UTC')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.sql.sources.partitionOverwriteMode', 'dynamic')
         .getOrCreate())
print('Spark', spark.version)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spark 4.0.5


In [2]:
restaurar('silver', 'logs')
for p in RUTAS.values():
    p.mkdir(parents=True, exist_ok=True)      # crea las carpetas que falten (vacías)

✓ restaurado: silver
✓ restaurado: logs


In [3]:
#la funcion de gold
# groupby(...).agg.(...): es mapreduce de la entrega 1. groupby define la clave y agg es el reduce. spark hace el shuffle por dentro
# repartition('event_date') antes de escribir: junta las filas de cada dia en una sola tarea, asi se escribe un archivo por dia. es la compactacion del diseño: bronce tenia cientos de archivos chicos, gold no
# industry y plan_tier dentro del mart: se desnormalizan porque cassandra no hace joins. si finops quiere ver el costo por industria, el dato tiene que estar en la misma tabla

import json, datetime
SILVER_EV = RUTAS['silver'] / 'usage_events'
GOLD_MART = RUTAS['gold'] / 'org_daily_usage_by_service'

def recalcular_fechas(fechas=None):
    """Recalcula el mart para las fechas indicadas (None = todas) y reemplaza esas particiones."""
    s = spark.read.parquet(str(SILVER_EV))
    if fechas is not None:
        s = s.filter(F.col('event_date').isin([str(f) for f in fechas]))
    s = s.dropDuplicates(['event_id'])          # garantía final contra duplicados (feedback E1)

    mart = (s.groupBy('org_id', 'event_date', 'service')
              .agg(F.round(F.sum('cost_usd_increment'), 4).alias('daily_cost_usd'),
                   F.sum('requests').alias('requests'),
                   F.round(F.sum('cpu_hours'), 4).alias('cpu_hours'),
                   F.round(F.sum('storage_gb_hours'), 4).alias('storage_gb_hours'),
                   F.sum('genai_tokens').alias('genai_tokens'),
                   F.round(F.sum('carbon_kg'), 4).alias('carbon_kg'),
                   F.count('*').alias('eventos'),
                   F.sum(F.col('flag_costo_anomalo').cast('int')).alias('eventos_costo_anomalo'),
                   F.sum(F.col('flag_pico_costo').cast('int')).alias('eventos_pico'),
                   F.first('industry').alias('industry'),
                   F.first('plan_tier').alias('plan_tier'))
              .withColumn('updated_at', F.current_timestamp()))

    (mart.repartition('event_date')                      # 1 archivo por día: evita archivos chicos
         .write.mode('overwrite').partitionBy('event_date').parquet(str(GOLD_MART)))
    return mart

In [4]:
#local = spark.read.parquet(str(RUTAS['silver'] / 'usage_events'))
#print('Silver LOCAL :', local.count(), 'eventos ·', local.select('event_date').distinct().count(), 'días')
#print('Silver DRIVE :', len(list((LAKE_DRIVE / 'silver' / 'usage_events').glob('event_date=*'))), 'carpetas de días')


In [5]:
# contruir el mart completo
mart = recalcular_fechas()           # primera vez: todas las fechas
g = spark.read.parquet(str(GOLD_MART))
resumen = g.agg(F.count('*').alias('filas'), F.round(F.sum('daily_cost_usd'), 2).alias('costo_total'),
                F.sum('eventos').alias('eventos'), F.countDistinct('event_date').alias('dias')).first().asDict()
resumen

{'filas': 11050, 'costo_total': 147433.98, 'eventos': 43200, 'dias': 60}

In [6]:
# validacion contra la entrega 1
# Valores del MapReduce manual (entrega 1, notebook 05): 11.050 claves · 147.433,98 USD · 43.200 eventos
assert resumen['filas'] == 11_050,            'Distinta cantidad de claves que en la entrega 1'
assert abs(resumen['costo_total'] - 147_433.98) < 0.05, 'Distinto costo total'
assert resumen['eventos'] == 43_200,          'Distinta cantidad de eventos'
print(' Spark reproduce exactamente el MapReduce de la entrega 1')
g.orderBy(F.desc('daily_cost_usd')).show(5, truncate=False)

 Spark reproduce exactamente el MapReduce de la entrega 1
+------------+-------+--------------+--------+---------+----------------+------------+---------+-------+---------------------+------------+----------+----------+--------------------------+----------+
|org_id      |service|daily_cost_usd|requests|cpu_hours|storage_gb_hours|genai_tokens|carbon_kg|eventos|eventos_costo_anomalo|eventos_pico|industry  |plan_tier |updated_at                |event_date|
+------------+-------+--------------+--------+---------+----------------+------------+---------+-------+---------------------+------------+----------+----------+--------------------------+----------+
|org_rixa11dp|genai  |340.5259      |390.0   |0.0473   |8.3086          |5646        |0.0797   |6      |0                    |1           |government|standard  |2026-11-15 22:43:32.529224|2025-08-24|
|org_c11ertj5|genai  |334.6767      |237.0   |NULL     |12.9349         |NULL        |NULL     |3      |0                    |1           |ret

In [7]:
# idempotencia y recalculo parcial

antes = g.agg(F.count('*'), F.round(F.sum('daily_cost_usd'), 2)).first()

recalcular_fechas()                                # 1) re-ejecución completa
recalcular_fechas(['2025-07-03', '2025-08-31'])    # 2) cierre D+1 simulado: solo 2 fechas

despues = spark.read.parquet(str(GOLD_MART)).agg(F.count('*'), F.round(F.sum('daily_cost_usd'), 2)).first()
print('antes  :', tuple(antes))
print('después:', tuple(despues))
assert tuple(antes) == tuple(despues), 'La re-ejecución cambió el mart'
print('Re-ejecutar (total o parcial) no duplica ni altera el mart')
!find "{GOLD_MART}" -name '*.parquet' | wc -l

antes  : (11050, 147433.98)
después: (11050, 147433.98)
Re-ejecutar (total o parcial) no duplica ni altera el mart
60


In [9]:
guardar_en_drive('gold', 'logs')

✓ sincronizado: gold
✓ sincronizado: logs


## Cierre · Gold: `org_daily_usage_by_service`

**Qué se hizo**
- Mart FinOps con grano **organización × día × servicio**: `daily_cost_usd`, `requests`, `cpu_hours`, `storage_gb_hours`, `genai_tokens`, `carbon_kg`, `eventos`, `eventos_costo_anomalo`, `eventos_pico`, más `industry` y `plan_tier` desnormalizados para el serving query-first en Cassandra.
- **Estrategia de actualización:** `recalcular_fechas(fechas)` recalcula días completos desde Silver y reemplaza solo esas particiones (overwrite dinámico). Es correcto ante eventos tardíos (se recalcula el día del evento) e idempotente ante reprocesos.
- **Dedupe por `event_id` antes de agregar:** garantía final contra duplicados, corrección derivada del feedback de la entrega 1.
- **Compactación:** `repartition('event_date')` → 1 archivo por día (60 archivos), frente a cientos en Bronze.

**Validación cruzada con la entrega 1**
| Métrica | MapReduce manual (E1) | Spark (E2) |
|---|---|---|
| Claves (org × día × servicio) | 11.050 | **11.050**  |
| Costo total (USD) | 147.433,98 | **147.433,98**  |
| Eventos | 43.200 | **43.200**  |

Dos implementaciones independientes producen el mismo resultado.

**Idempotencia**
| Operación | Filas | Costo total |
|---|---|---|
| Construcción inicial | 11.050 | 147.433,98 |
| Re-ejecución completa + recálculo parcial (2 fechas, cierre D+1 simulado) | 11.050 | 147.433,98 |

**Lectura de negocio:** los 5 días-servicio más caros contienen un evento pico (`eventos_pico = 1`) y 3 son de GenAI, consistente con la concentración de costo detectada en la entrega 1.

**Hallazgo operativo:** la primera corrida tomó una copia incompleta de Silver (17 de 60 días) porque se inició antes de que terminara la sincronización del notebook anterior; las validaciones automáticas (`assert`) lo detectaron. Correcciones: un notebook a la vez, restauración y sincronización por zona (`restaurar` / `guardar_en_drive` con rutas explícitas).